# 30年分年度PAWN敏感性分析（按气候区分组）

本notebook对1991-2020年每一年的数据，分别在4个气候区（HW、HD、CW、CD）中进行PAWN敏感性分析。

**分析设置：**
- 时间范围：1991-2020（30年）
- 气候区：HW（热湿）、HD（热干）、CW（冷湿）、CD（冷干）
- 敏感性方法：PAWN
- 输入变量：precipitation, lai
- 输出变量：evapotrans, tran, evspsblveg, evspsblsoi

## 0. 导入库和配置

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy.stats import ks_2samp
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# 设置绘图样式
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

print("库导入成功！")

库导入成功！


## 1. 读取数据和气候区标签

In [2]:
# 读取已打好气候区标签的30年数据（包含time列）
classic_path = Path('data/preprocessed/preprocessed_with_zones/classic_with_climate_zones_filtered_30y.csv')
lpj_path = Path('data/preprocessed/preprocessed_with_zones/lpj_guess_with_climate_zones_filtered_30y.csv')

print("读取已打好气候区标签的30年数据...")
classic_df = pd.read_csv(classic_path)
lpj_df = pd.read_csv(lpj_path)

print(f"\nCLASSIC数据: {len(classic_df):,} 行")
print(f"LPJ-GUESS数据: {len(lpj_df):,} 行")

print("\n时间范围:")
print(f"  CLASSIC: {classic_df['time'].min()} - {classic_df['time'].max()}")
print(f"  LPJ-GUESS: {lpj_df['time'].min()} - {lpj_df['time'].max()}")

print("\n气候区分布:")
print("CLASSIC:")
print(classic_df['climate_zone'].value_counts())
print("\nLPJ-GUESS:")
print(lpj_df['climate_zone'].value_counts())

读取已打好气候区标签的30年数据...

CLASSIC数据: 1,715,376 行
LPJ-GUESS数据: 1,659,234 行

时间范围:
  CLASSIC: 1991 - 2020
  LPJ-GUESS: 1991 - 2020

气候区分布:
CLASSIC:
climate_zone
CW         629799
CD         384477
HW         366807
HD         333276
Unknown      1017
Name: count, dtype: int64

LPJ-GUESS:
climate_zone
CW         635027
HW         370909
HD         338383
CD         310410
Unknown      4505
Name: count, dtype: int64


## 2. 定义PAWN敏感性分析函数

In [3]:
def calculate_pawn_index(df, input_var, output_var, n_bins=10):
    """
    计算PAWN敏感性指数
    
    PAWN方法使用Kolmogorov-Smirnov统计量来衡量条件分布与无条件分布的差异
    
    参数:
        df: 数据框
        input_var: 输入变量名
        output_var: 输出变量名
        n_bins: 分箱数量（默认10）
    
    返回:
        dict: {'median': PAWN中位数, 'max': PAWN最大值}
    """
    # 无条件输出分布
    y_unconditional = df[output_var].values
    
    # 将输入变量分箱
    try:
        bins = pd.qcut(df[input_var], q=n_bins, duplicates='drop')
    except ValueError:
        bins = pd.cut(df[input_var], bins=n_bins)
    
    # 计算每个bin的KS统计量
    ks_stats = []
    for bin_label in bins.cat.categories:
        y_conditional = df.loc[bins == bin_label, output_var].values
        if len(y_conditional) < 10:
            continue
        ks_stat, p_value = ks_2samp(y_unconditional, y_conditional)
        ks_stats.append(ks_stat)
    
    if len(ks_stats) == 0:
        return {'median': 0, 'max': 0}
    
    return {
        'median': np.median(ks_stats),
        'max': np.max(ks_stats)
    }

print("✓ PAWN函数定义完成")

✓ PAWN函数定义完成


## 3. 执行30年分年度PAWN分析

In [4]:
# 定义变量
input_vars = ['precipitation', 'lai']
output_vars = ['evapotrans', 'tran', 'evspsblveg', 'evspsblsoi']
climate_zones = ['HW', 'HD', 'CW', 'CD']
years = sorted(classic_df['time'].unique())

print(f"分析配置:")
print(f"  年份数: {len(years)} ({years[0]}-{years[-1]})")
print(f"  气候区: {climate_zones}")
print(f"  输入变量: {input_vars}")
print(f"  输出变量: {output_vars}")
print(f"  总分析数: {len(years)} × 4气候区 × 2输入 × 4输出 × 2模型 = {len(years)*4*2*4*2}")

分析配置:
  年份数: 30 (1991-2020)
  气候区: ['HW', 'HD', 'CW', 'CD']
  输入变量: ['precipitation', 'lai']
  输出变量: ['evapotrans', 'tran', 'evspsblveg', 'evspsblsoi']
  总分析数: 30 × 4气候区 × 2输入 × 4输出 × 2模型 = 1920


In [5]:
# ============================================================================
# CLASSIC模型：PAWN分析
# ============================================================================
print("="*80)
print("CLASSIC模型：30年分年度PAWN分析")
print("="*80)

classic_pawn_results = []

for year in tqdm(years, desc="PAWN - 年份进度"):
    # 筛选该年的数据
    year_data = classic_df[classic_df['time'] == year].copy()
    
    for zone in climate_zones:
        # 筛选该气候区的数据
        zone_data = year_data[year_data['climate_zone'] == zone].copy()
        
        if len(zone_data) < 50:  # 样本数太少则跳过
            continue
        
        for output_var in output_vars:
            for input_var in input_vars:
                # 计算PAWN指数
                pawn_result = calculate_pawn_index(zone_data, input_var, output_var)
                
                # 保存结果
                classic_pawn_results.append({
                    'model': 'CLASSIC',
                    'year': year,
                    'climate_zone': zone,
                    'input_var': input_var,
                    'output_var': output_var,
                    'pawn_median': pawn_result['median'],
                    'pawn_max': pawn_result['max'],
                    'n_samples': len(zone_data)
                })

classic_pawn_df = pd.DataFrame(classic_pawn_results)
print(f"\n✓ CLASSIC PAWN分析完成，共{len(classic_pawn_df):,}条结果")
print(f"\n前10行:")
print(classic_pawn_df.head(10))

# 保存PAWN结果
output_dir = Path('output/30y_PAWN')
output_dir.mkdir(parents=True, exist_ok=True)
classic_pawn_df.to_csv(output_dir / 'classic_30y_pawn_results.csv', index=False)
print(f"\n✓ PAWN结果已保存到: {output_dir / 'classic_30y_pawn_results.csv'}")

CLASSIC模型：30年分年度PAWN分析


PAWN - 年份进度: 100%|██████████| 30/30 [00:37<00:00,  1.25s/it]


✓ CLASSIC PAWN分析完成，共960条结果

前10行:
     model  year climate_zone      input_var  output_var  pawn_median  \
0  CLASSIC  1991           HW  precipitation  evapotrans     0.281586   
1  CLASSIC  1991           HW            lai  evapotrans     0.382286   
2  CLASSIC  1991           HW  precipitation        tran     0.228284   
3  CLASSIC  1991           HW            lai        tran     0.396583   
4  CLASSIC  1991           HW  precipitation  evspsblveg     0.322884   
5  CLASSIC  1991           HW            lai  evspsblveg     0.440094   
6  CLASSIC  1991           HW  precipitation  evspsblsoi     0.226265   
7  CLASSIC  1991           HW            lai  evspsblsoi     0.400410   
8  CLASSIC  1991           HD  precipitation  evapotrans     0.492088   
9  CLASSIC  1991           HD            lai  evapotrans     0.475859   

   pawn_max  n_samples  
0  0.478173      12287  
1  0.630850      12287  
2  0.434032      12287  
3  0.703037      12287  
4  0.512701      12287  
5  0.758614

In [6]:
# ============================================================================
# LPJ-GUESS模型：PAWN分析
# ============================================================================
print("="*80)
print("LPJ-GUESS模型：30年分年度PAWN分析")
print("="*80)

lpj_pawn_results = []

for year in tqdm(years, desc="PAWN - 年份进度"):
    # 筛选该年的数据
    year_data = lpj_df[lpj_df['time'] == year].copy()
    
    for zone in climate_zones:
        # 筛选该气候区的数据
        zone_data = year_data[year_data['climate_zone'] == zone].copy()
        
        if len(zone_data) < 50:  # 样本数太少则跳过
            continue
        
        for output_var in output_vars:
            for input_var in input_vars:
                # 计算PAWN指数
                pawn_result = calculate_pawn_index(zone_data, input_var, output_var)
                
                # 保存结果
                lpj_pawn_results.append({
                    'model': 'LPJ-GUESS',
                    'year': year,
                    'climate_zone': zone,
                    'input_var': input_var,
                    'output_var': output_var,
                    'pawn_median': pawn_result['median'],
                    'pawn_max': pawn_result['max'],
                    'n_samples': len(zone_data)
                })

lpj_pawn_df = pd.DataFrame(lpj_pawn_results)
print(f"\n✓ LPJ-GUESS PAWN分析完成，共{len(lpj_pawn_df):,}条结果")
print(f"\n前10行:")
print(lpj_pawn_df.head(10))

# 保存PAWN结果
lpj_pawn_df.to_csv(output_dir / 'lpj_30y_pawn_results.csv', index=False)
print(f"\n✓ PAWN结果已保存到: {output_dir / 'lpj_30y_pawn_results.csv'}")

LPJ-GUESS模型：30年分年度PAWN分析


PAWN - 年份进度: 100%|██████████| 30/30 [00:38<00:00,  1.29s/it]


✓ LPJ-GUESS PAWN分析完成，共960条结果

前10行:
       model  year climate_zone      input_var  output_var  pawn_median  \
0  LPJ-GUESS  1991           HW  precipitation  evapotrans     0.297161   
1  LPJ-GUESS  1991           HW            lai  evapotrans     0.461227   
2  LPJ-GUESS  1991           HW  precipitation        tran     0.242009   
3  LPJ-GUESS  1991           HW            lai        tran     0.439445   
4  LPJ-GUESS  1991           HW  precipitation  evspsblveg     0.322072   
5  LPJ-GUESS  1991           HW            lai  evspsblveg     0.507211   
6  LPJ-GUESS  1991           HW  precipitation  evspsblsoi     0.140392   
7  LPJ-GUESS  1991           HW            lai  evspsblsoi     0.114267   
8  LPJ-GUESS  1991           HD  precipitation  evapotrans     0.491776   
9  LPJ-GUESS  1991           HD            lai  evapotrans     0.516184   

   pawn_max  n_samples  
0  0.500235      12463  
1  0.731549      12463  
2  0.443515      12463  
3  0.724173      12463  
4  0.591001 

In [7]:
# 合并两个模型的结果
all_pawn_df = pd.concat([classic_pawn_df, lpj_pawn_df], ignore_index=True)

print(f"合并结果统计:")
print(f"  总记录数: {len(all_pawn_df):,}")
print(f"  CLASSIC: {len(classic_pawn_df):,}")
print(f"  LPJ-GUESS: {len(lpj_pawn_df):,}")

# 导出合并结果
all_pawn_df.to_csv(output_dir / 'all_30y_pawn_results.csv', index=False)
print(f"\n✓ 合并结果已导出: {output_dir / 'all_30y_pawn_results.csv'}")

合并结果统计:
  总记录数: 1,920
  CLASSIC: 960
  LPJ-GUESS: 960

✓ 合并结果已导出: output/30y_PAWN/all_30y_pawn_results.csv


## 4. 可视化：PAWN指数时间序列图

In [8]:
def plot_pawn_timeseries_by_zone(df, model_name, output_var, metric='pawn_median'):
    """
    绘制PAWN指数时间序列图（按气候区分组）
    
    每个子图代表一个气候区，显示2个输入变量的时间趋势
    """
    fig, axes = plt.subplots(2, 2, figsize=(18, 12))
    axes = axes.flatten()
    
    zone_colors = {
        'HW': '#FF6B6B',
        'HD': '#FFD93D',
        'CW': '#6BCB77',
        'CD': '#4D96FF'
    }
    
    zone_names = {
        'HW': 'Hot-Wet',
        'HD': 'Hot-Dry',
        'CW': 'Cold-Wet',
        'CD': 'Cold-Dry'
    }
    
    input_colors = {
        'precipitation': '#3498db',
        'lai': '#e74c3c'
    }
    
    for i, zone in enumerate(climate_zones):
        ax = axes[i]
        
        # 筛选该气候区和输出变量的数据
        zone_data = df[(df['climate_zone'] == zone) & 
                      (df['output_var'] == output_var)]
        
        if len(zone_data) == 0:
            ax.text(0.5, 0.5, f'No data for {zone}',
                   ha='center', va='center', fontsize=14)
            ax.axis('off')
            continue
        
        # 绘制每个输入变量的时间序列
        for input_var in input_vars:
            input_data = zone_data[zone_data['input_var'] == input_var].sort_values('year')
            
            if len(input_data) > 0:
                ax.plot(input_data['year'], input_data[metric],
                       marker='o', linewidth=2, markersize=4,
                       color=input_colors[input_var],
                       label=input_var.upper(),
                       alpha=0.8)
        
        ax.set_xlabel('Year', fontsize=11, fontweight='bold')
        ax.set_ylabel('PAWN Index (median)', fontsize=11, fontweight='bold')
        ax.set_title(f'{zone} - {zone_names[zone]}',
                    fontsize=13, fontweight='bold',
                    color=zone_colors[zone])
        ax.grid(True, alpha=0.3)
        ax.legend(loc='best', fontsize=10)
        
        # 设置x轴刻度（每5年）
        years_range = range(int(input_data['year'].min()), 
                          int(input_data['year'].max()) + 1, 5)
        ax.set_xticks(years_range)
        ax.tick_params(axis='x', rotation=45)
    
    fig.suptitle(f'{model_name} - PAWN Time Series for {output_var.upper()}',
                 fontsize=16, fontweight='bold', y=0.995)
    plt.tight_layout()
    
    return fig

print("✓ 时间序列绘图函数定义完成")

✓ 时间序列绘图函数定义完成


In [ ]:
# 为每个输出变量绘制CLASSIC模型的时间序列图
print("生成CLASSIC模型PAWN时间序列图...")

for output_var in output_vars:
    fig = plot_pawn_timeseries_by_zone(
        classic_pawn_df,
        'CLASSIC',
        output_var,
        metric='pawn_median'
    )
    
    # 保存图片
    fig_path = output_dir / f'classic_pawn_timeseries_{output_var}.png'
    fig.savefig(fig_path, dpi=300, bbox_inches='tight')
    plt.close(fig)
    print(f"  ✓ {fig_path.name}")

print("\n✓ CLASSIC模型时间序列图生成完成")

In [ ]:
# 为每个输出变量绘制LPJ-GUESS模型的时间序列图
print("生成LPJ-GUESS模型PAWN时间序列图...")

for output_var in output_vars:
    fig = plot_pawn_timeseries_by_zone(
        lpj_pawn_df,
        'LPJ-GUESS',
        output_var,
        metric='pawn_median'
    )
    
    # 保存图片
    fig_path = output_dir / f'lpj_pawn_timeseries_{output_var}.png'
    fig.savefig(fig_path, dpi=300, bbox_inches='tight')
    plt.close(fig)
    print(f"  ✓ {fig_path.name}")

print("\n✓ LPJ-GUESS模型时间序列图生成完成")

## 5. 可视化：PAWN指数热图

In [ ]:
def plot_pawn_heatmap(df, model_name, input_var, output_var, metric='pawn_median'):
    """
    绘制PAWN指数热图（年份 × 气候区）
    
    参数:
        df: 结果数据框
        model_name: 模型名称
        input_var: 输入变量
        output_var: 输出变量
        metric: 指标（pawn_median或pawn_max）
    """
    # 筛选数据
    subset = df[(df['input_var'] == input_var) & 
               (df['output_var'] == output_var)]
    
    # 透视表：年份 × 气候区
    pivot_data = subset.pivot(index='year', 
                             columns='climate_zone', 
                             values=metric)
    
    # 按气候区顺序排列
    pivot_data = pivot_data[climate_zones]
    
    # 绘制热图
    fig, ax = plt.subplots(figsize=(10, 12))
    
    sns.heatmap(pivot_data, 
                cmap='YlOrRd', 
                annot=False,
                fmt='.3f',
                cbar_kws={'label': 'PAWN Index (median)'},
                linewidths=0.5,
                ax=ax)
    
    ax.set_xlabel('Climate Zone', fontsize=12, fontweight='bold')
    ax.set_ylabel('Year', fontsize=12, fontweight='bold')
    ax.set_title(f'{model_name} - PAWN Heatmap\n'
                f'Input: {input_var.upper()}, Output: {output_var.upper()}',
                fontsize=14, fontweight='bold', pad=15)
    
    plt.tight_layout()
    return fig

print("✓ 热图绘制函数定义完成")

In [ ]:
# 为每个输入-输出组合绘制热图（CLASSIC模型）
print("生成CLASSIC模型PAWN热图...")

for input_var in input_vars:
    for output_var in output_vars:
        fig = plot_pawn_heatmap(
            classic_pawn_df,
            'CLASSIC',
            input_var,
            output_var,
            metric='pawn_median'
        )
        
        # 保存图片
        fig_path = output_dir / f'classic_pawn_heatmap_{input_var}_{output_var}.png'
        fig.savefig(fig_path, dpi=300, bbox_inches='tight')
        plt.close(fig)
        print(f"  ✓ {fig_path.name}")

print("\n✓ CLASSIC模型热图生成完成")

In [ ]:
# 为每个输入-输出组合绘制热图（LPJ-GUESS模型）
print("生成LPJ-GUESS模型PAWN热图...")

for input_var in input_vars:
    for output_var in output_vars:
        fig = plot_pawn_heatmap(
            lpj_pawn_df,
            'LPJ-GUESS',
            input_var,
            output_var,
            metric='pawn_median'
        )
        
        # 保存图片
        fig_path = output_dir / f'lpj_pawn_heatmap_{input_var}_{output_var}.png'
        fig.savefig(fig_path, dpi=300, bbox_inches='tight')
        plt.close(fig)
        print(f"  ✓ {fig_path.name}")

print("\n✓ LPJ-GUESS模型热图生成完成")

## 6. 统计汇总

In [ ]:
# 计算30年平均PAWN指数（按气候区和变量组合）
summary_stats = all_pawn_df.groupby(
    ['model', 'climate_zone', 'input_var', 'output_var']
).agg({
    'pawn_median': ['mean', 'std', 'min', 'max'],
    'pawn_max': ['mean', 'std', 'min', 'max'],
    'n_samples': 'mean'
}).round(4)

# 展平多级列名
summary_stats.columns = ['_'.join(col).strip() for col in summary_stats.columns.values]
summary_stats = summary_stats.reset_index()

print("30年平均PAWN指数统计汇总:")
print(summary_stats.head(20))

# 导出统计汇总
summary_stats.to_csv(output_dir / '30y_pawn_summary_statistics.csv', index=False)
print(f"\n✓ 统计汇总已导出: {output_dir / '30y_pawn_summary_statistics.csv'}")

## 7. 任务完成总结

In [ ]:
print("="*80)
print("30年分年度PAWN敏感性分析完成！")
print("="*80)

print("\n分析概况:")
print(f"  时间范围: {years[0]}-{years[-1]} ({len(years)}年)")
print(f"  气候区: {', '.join(climate_zones)}")
print(f"  模型: CLASSIC, LPJ-GUESS")
print(f"  输入变量: {', '.join(input_vars)}")
print(f"  输出变量: {', '.join(output_vars)}")

print("\n结果统计:")
print(f"  CLASSIC结果: {len(classic_pawn_df):,} 条")
print(f"  LPJ-GUESS结果: {len(lpj_pawn_df):,} 条")
print(f"  总计: {len(all_pawn_df):,} 条")

print("\n输出文件:")
print(f"  目录: {output_dir}")
print("  CSV文件:")
print("    - classic_30y_pawn_results.csv")
print("    - lpj_30y_pawn_results.csv")
print("    - all_30y_pawn_results.csv")
print("    - 30y_pawn_summary_statistics.csv")

print("\n可视化图表:")
print(f"  PAWN时间序列图: {len(output_vars)} × 2模型 = {len(output_vars)*2} 张")
print(f"  PAWN热图: {len(input_vars)} × {len(output_vars)} × 2模型 = {len(input_vars)*len(output_vars)*2} 张")

print("\n✓ 分析完成！")